# 2.1 Intro and API basics: requests, multi-turn chat, system prompts

Step 2, lesson 1 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api) (Anthropic Academy).
Exam domains: **4 Prompt Engineering & Structured Output** (system prompts) and **5 Context Management & Reliability** (the API is stateless, so *you* manage the conversation).

What this notebook covers, in the course's order:

1. Which Claude: the model family
2. Getting an API key and keeping it out of the code
3. Making a first request, and reading the response
4. Multi-turn conversations: the API remembers nothing, so we resend the history
5. A small chat loop
6. System prompts: telling Claude *how* to behave
7. Temperature, and what changed on current models

Four exercises along the way let you practise each part. For each exercise: write your code in the *Your turn* cell and run it, then run the *Check* cell. The *Solution* cell sits in between: open the hint first, and skip the solution unless you're stuck.

> **Model used here:** `claude-sonnet-5-5`. The course notebooks name an older model; swap the `model` variable to try another one.

## 1. Which Claude? The model family

The course opens with an overview of Claude's models. The idea hasn't changed, only the names: Claude is a family of models, and you trade capability against speed and price.

| Model | API id | Good for | Price per million tokens (input / output) |
|---|---|---|---|
| Claude Haiku 5.5 | `claude-haiku-5-5` | fast, cheap, high-volume work: classifying, routing, extracting, sub-agent tasks | $0.10 / $0.50 |
| Claude Sonnet 5.5 | `claude-sonnet-5-5` | the everyday balance of quality, speed and cost | $2 / $10 |
| Claude Opus 5.5 | `claude-opus-5-5` | long, complex agentic and coding work | $4 / $20 |
| Claude Fable 5.2 | `claude-fable-5-2` | the most capable: demanding reasoning, multi-day tasks | $8 / $40 |

All four read up to 1 million tokens of context. Prices as of November 2026, from the [models overview](https://platform.claude.com/docs/en/about-claude/models/overview); Haiku's applies to prompts up to 100K tokens.

> **Changed since the course.** The course compares an older generation. The rule of thumb is the same: start with the smallest model that passes your evals (lesson 2.3), and move up only where it falls short.

## 2. API key and client

You create a key in the [Claude Console](https://platform.claude.com) under **Keys**. A key is a password for your account's credits, so:

- it lives in a `.env` file at the repo root (`ANTHROPIC_API_KEY=sk-ant-...`), which is in `.gitignore`;
- the code never contains it and never prints it;
- `load_dotenv()` copies it into the environment, and `Anthropic()` reads it from there.

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()          # finds the .env file in this folder or a parent folder
client = Anthropic()   # reads ANTHROPIC_API_KEY from the environment
model = "claude-sonnet-5-5"

Which models can your key use? The Models API lists them. It's free: no tokens are used.

In [2]:
for m in client.models.list():
    print(f"{m.id:20} {m.display_name}")

claude-fable-5-2     Claude Fable 5.2
claude-haiku-5-5     Claude Haiku 5.5
claude-sonnet-5-5    Claude Sonnet 5.5
claude-opus-5-5      Claude Opus 5.5
claude-fable-5-1     Claude Fable 5.1
claude-opus-5        Claude Opus 5
claude-sonnet-5      Claude Sonnet 5
claude-fable-5       Claude Fable 5
claude-opus-4-8      Claude Opus 4.8
claude-opus-4-7      Claude Opus 4.7
claude-sonnet-4-6    Claude Sonnet 4.6
claude-opus-4-6      Claude Opus 4.6
claude-opus-4-5-20251101 Claude Opus 4.5
claude-haiku-4-5-20251001 Claude Haiku 4.5
claude-sonnet-4-5-20250929 Claude Sonnet 4.5


## 3. A first request

Every call goes to one endpoint, **Messages** (`client.messages.create`). Three arguments are required:

| Argument | What it is |
|---|---|
| `model` | which Claude to use |
| `max_tokens` | a hard cap on how much Claude may write (a safety limit, not a target length) |
| `messages` | the conversation so far: a list of `{"role": "user" or "assistant", "content": ...}` |

In [3]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": "What is quantum computing? Answer in one sentence."}],
)
message

Message(id='msg_011Ch67cNMeXUn9PsA96oCWq', container=None, content=[TextBlock(citations=None, text='Quantum computing is a type of computing that uses quantum-mechanical phenomena like superposition and entanglement, through units called qubits, to solve certain problems much faster than classical computers can.', type='text')], diagnostics=None, model='claude-sonnet-5-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=25, output_tokens=63, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))

The answer is a `Message` object. The parts worth knowing:

- `content`: a **list of blocks**. Text is in blocks of type `text`; later lessons add `tool_use` and `thinking` blocks.
- `stop_reason`: why Claude stopped (`end_turn` = finished, `max_tokens` = cut off by your cap, `tool_use` = wants to call a tool).
- `usage`: input and output tokens, which is what you pay for.

> **Changed since the course.** The course reads the answer with `message.content[0].text`. On current models the first block can be a `thinking` block (adaptive thinking is on by default), so pick the first block whose type is `text`.

In [4]:
def text_of(message):
    """Return the text of the first text block in a response ("" if there is none)."""
    return next((block.text for block in message.content if block.type == "text"), "")

print([block.type for block in message.content], message.stop_reason)
print(message.usage.input_tokens, "input tokens,", message.usage.output_tokens, "output tokens")
print(text_of(message))

['text'] end_turn
25 input tokens, 63 output tokens
Quantum computing is a type of computing that uses quantum-mechanical phenomena like superposition and entanglement, through units called qubits, to solve certain problems much faster than classical computers can.


### Exercise 1 · Spot an answer that was cut off

`max_tokens` is a hard cap. When Claude reaches it, the answer simply stops, mid-sentence if need be, and the only sign is `stop_reason == "max_tokens"`. Code that doesn't check it passes half an answer on as if it were complete (a Domain 5 reliability point).

**Task.**

1. Write `was_cut_off(message)`, returning `True` when the response stopped because of `max_tokens`.
2. Make a request with `max_tokens=30` that asks for something long (say, "Explain how a rainbow forms."), and keep the response in `short`. Use Haiku 5.5 (`"claude-haiku-5-5"`) for it: it's the cheapest.

**Done when** the check cell prints ✅.

In [5]:
# Your turn
def was_cut_off(message):
    ...  # True when Claude stopped because it reached max_tokens


short = None  # replace with client.messages.create(...), model "claude-haiku-5-5", max_tokens=30

<details><summary><b>Hint</b></summary>

The response has a `stop_reason` attribute: compare it with the string `"max_tokens"`. For the request, copy the first request above and change `model`, `max_tokens` and the question.

</details>

**Solution** (skip it unless you're stuck)

In [6]:
# Solution
def was_cut_off(message):
    return message.stop_reason == "max_tokens"


short = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=30,
    messages=[{"role": "user", "content": "Explain how a rainbow forms."}],
)
print(repr(text_of(short)))
print(short.stop_reason, "|", short.usage.output_tokens, "output tokens")

'A rainbow forms through the interaction of sunlight with water droplets in the air, primarily through'
max_tokens | 30 output tokens


In [7]:
# Check
if short is None or was_cut_off(short) is None:
    print("❌ Not yet: write was_cut_off and make the short request.")
elif was_cut_off(short) and not was_cut_off(message):
    print("✅ The 30-token answer is flagged as cut off, and the first answer (stop_reason 'end_turn') is not.")
else:
    print("❌ Not yet: was_cut_off(short) should be True, and was_cut_off(message) False.")

✅ The 30-token answer is flagged as cut off, and the first answer (stop_reason 'end_turn') is not.


## 4. Multi-turn conversations

The API is **stateless**: it keeps nothing between calls. To have a conversation, you keep the list of messages yourself and send *all of it* every time, alternating `user` and `assistant`.

The course builds three small helpers for this, and the rest of the course uses them:

In [8]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, max_tokens=1000):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return text_of(message)

In [9]:
messages = []

add_user_message(messages, "Define quantum computing in one sentence.")
answer = chat(messages)
add_assistant_message(messages, answer)
print("Claude:", answer, "\n")

# A follow-up that only makes sense with the history
add_user_message(messages, "Write another sentence that builds on that one.")
answer = chat(messages)
add_assistant_message(messages, answer)
print("Claude:", answer)

Claude: Quantum computing is a type of computation that uses quantum-mechanical phenomena such as superposition and entanglement to process information with quantum bits (qubits), allowing it to solve certain problems much faster than classical computers can. 



Claude: While classical computers store information as bits that are strictly 0 or 1, qubits can exist in a blend of both states at once, which lets quantum algorithms explore many possibilities simultaneously and makes them promising for tasks like cryptography, drug discovery, and complex optimization.


Without the history, the second question has no meaning. Same question, sent alone:

In [10]:
print(chat([{"role": "user", "content": "Write another sentence that builds on that one."}]))

I don't see a previous sentence in our conversation, so I'm missing what to build on. Could you share it?

If you'd like, I can also start with one of my own. For example: *The old lighthouse stood alone at the edge of the cliff, its light long extinguished.* Then I could add: *Yet every night, fishermen swore they could still see a faint glow sweeping across the waves.*

Just paste your sentence, and I'll continue from there.


**Why this matters for the exam (Domain 5).** Because you resend the whole history on every call, a long conversation costs more on every turn and eventually fills the context window. Everything in Domain 5 (summarising old turns, trimming tool results, keeping "case facts" in a fixed block) is a way of managing this list that *you* own.

## 5. A small chat loop

The course's exercise is a loop: read a line from the user, add it, ask Claude, add the answer, repeat. In Jupyter you would use `input()`; here a fixed list of lines stands in for the person typing, so the notebook runs from top to bottom.

In [11]:
messages = []
user_lines = ["Hi! My name is Sam.", "I'm learning the Claude API.", "What's my name, and what am I learning?"]

for line in user_lines:            # interactive version: while True: line = input("> ")
    add_user_message(messages, line)
    answer = chat(messages, max_tokens=300)
    add_assistant_message(messages, answer)
    print("You:   ", line)
    print("Claude:", answer, "\n")

You:    Hi! My name is Sam.
Claude: Hi Sam! Nice to meet you. How can I help you today? 



You:    I'm learning the Claude API.
Claude: That's a great thing to learn, Sam! Where are you in the process? A few common starting points:

- **Getting set up:** creating an API key, installing the SDK (Python or TypeScript), and making your first request
- **Core concepts:** the Messages API, system prompts, roles, tokens, and model selection
- **Going further:** streaming responses, tool use (function calling), vision, prompt caching, and structured outputs
- **Building something specific:** a chatbot, a document summarizer, an agent, etc.

For reference, here's a minimal Python example:

```python
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from your environment

message = client.messages.create(
    model="claude-sonnet-4-20250 



You:    What's my name, and what am I learning?
Claude: Your name is Sam, and you're learning the Claude API.

My last message got cut off in the middle of the code example. Want me to resend the full snippet, or should we start with a specific part, like setup, system prompts, or streaming? 



Look at Claude's second answer above: it stops in the middle of a code example, because `max_tokens=300` cut it off. In its next answer Claude even says so, because the cut-off reply is in the history. The loop itself never noticed.

### Exercise 2 · Your own chat, with a cut-off warning

This is the course's chat exercise, with the check from exercise 1 added.

**Task.** Write `chat_turn(messages, user_text, max_tokens=300)`, which:

1. adds `user_text` to `messages` as a user message,
2. calls the API with the whole list,
3. adds Claude's reply as an assistant message,
4. returns the reply text, followed by `"\n[cut off: max_tokens reached]"` when the reply was cut off.

Then run the chat cell after the solution. With `INTERACTIVE = True` it asks you for each line (VS Code opens an input box at the top; an empty line ends the chat). With `False` it plays three fixed lines.

**Done when** the check cell prints ✅.

In [12]:
# Your turn
def chat_turn(messages, user_text, max_tokens=300):
    ...  # add the user message, call the API, add the reply, return its text (plus a warning if cut off)

<details><summary><b>Hint</b></summary>

Reuse the helpers: `add_user_message`, then `client.messages.create(model=model, max_tokens=max_tokens, messages=messages)`, then `text_of(...)` and `add_assistant_message`. You need the whole response, not just its text, to call `was_cut_off` from exercise 1.

</details>

**Solution** (skip it unless you're stuck)

In [13]:
# Solution
def chat_turn(messages, user_text, max_tokens=300):
    add_user_message(messages, user_text)
    response = client.messages.create(model=model, max_tokens=max_tokens, messages=messages)
    reply = text_of(response)
    add_assistant_message(messages, reply)
    if was_cut_off(response):
        reply += "\n[cut off: max_tokens reached]"
    return reply

In [14]:
# The chat
INTERACTIVE = False   # True: type your own lines (an empty line ends the chat)

chat_history = []
script = ["Hi! I'm Alex, and I'm learning the Claude API.",
          "Give me three tips for writing good prompts.",
          "What's my name?"]


def next_line():
    if INTERACTIVE:
        return input("You: ")
    return script.pop(0) if script else ""


while line := next_line():
    print("You:   ", line)
    print("Claude:", chat_turn(chat_history, line, max_tokens=150), "\n")

You:    Hi! I'm Alex, and I'm learning the Claude API.


Claude: Hi Alex, nice to meet you! Learning the Claude API is a good project, and I'm happy to help.

To point you in the right direction, it would help to know:

- **Your background:** Are you new to APIs in general, or just to Claude's?
- **Your language:** Python, Typ
[cut off: max_tokens reached] 

You:    Give me three tips for writing good prompts.


Claude: Here are three tips that make a big difference:

1. **Be specific about what you want.** Vague prompts get vague answers. Instead of "Summarize this article," try "Summarize this article in 3 bullet points for a busy executive, focusing on financial implications." Include the audience, format, length, and tone when they matter.

2. **
[cut off: max_tokens reached] 

You:    What's my name?


Claude: Your name is Alex, as you mentioned in your first message. Is there anything else you'd like to know about prompting or the Claude API? 



In [15]:
# Check
roles = [m["role"] for m in chat_history]
alternating = len(roles) >= 4 and roles == ["user", "assistant"] * (len(roles) // 2)
remembers = INTERACTIVE or "Alex" in chat_history[-1]["content"]
if alternating and remembers:
    print(f"✅ {len(roles)} messages, alternating user and assistant, and Claude still knows your name: "
          "it saw the whole conversation on every turn.")
else:
    print("❌ Not yet: chat_history should alternate user, assistant, user, assistant, ..., "
          "and the last answer should know the name.")

✅ 6 messages, alternating user and assistant, and Claude still knows your name: it saw the whole conversation on every turn.


## 6. System prompts

A **system prompt** sets the role and rules for the whole conversation. It is a separate `system` argument, not a message. The course's example is a maths tutor who should guide, not give away answers.

In [16]:
system = """You are a patient math tutor.
Do not directly answer a student's question.
Guide them to a solution step by step, with one hint at a time."""

question = [{"role": "user", "content": "How do I solve 5x + 2 = 3 for x?"}]

print("--- without a system prompt ---")
print(chat(question, max_tokens=400))
print("\n--- with the tutor system prompt ---")
print(chat(question, system=system, max_tokens=400))

--- without a system prompt ---


**Step 1: Subtract 2 from both sides**

5x + 2 − 2 = 3 − 2

5x = 1

**Step 2: Divide both sides by 5**

x = 1/5

**Check:** Substitute x = 1/5 back into the original equation:

5(1/5) + 2 = 1 + 2 = 3 ✓

**Answer: x = 1/5** (or 0.2)

--- with the tutor system prompt ---


Good question! Let's work through it together, one step at a time.

The goal is to get **x** alone on one side of the equation. Right now, x is being multiplied by 5, and then 2 is being added to that.

**First hint:** To undo the operations, we usually work backward. Looking at the equation 5x + 2 = 3, which operation is "attached" to the 5x term on the left side, and what could you do to *both sides* of the equation to cancel it out?

Give that a try and tell me what you get!


**Exam angle (Domain 4).** A system prompt is the right first tool for *tone, role and style*. But it is guidance: Claude follows it very reliably, not with certainty. When a rule **must always hold** (never refund above a limit, always verify identity first), the guide's answer is to enforce it in code, for example a check before a tool runs, rather than adding more emphatic wording to the prompt.

### Exercise 3 · A system prompt you can check

A system prompt is easy to write and easy to get *almost* right. The way to know is to check the output in code, which is where lesson 2.3 (evals) starts.

**Task.** Write `study_system`, a system prompt for a study buddy that answers *every* question with exactly three bullet points (lines that start with `- `), each under 20 words, with no introduction or closing line. The check cell asks two questions with it and tests the format.

**Done when** both answers pass. If one fails, change the prompt and run the check again: that loop is prompt engineering (lesson 2.4).

In [17]:
# Your turn
study_system = """..."""

<details><summary><b>Hint</b></summary>

Say what to do, not only what to avoid, and say it exactly: "exactly three bullet points", "each line starts with `- `", "nothing before or after the list". Claude follows a precise description of the format better than a vague one ("be brief").

</details>

**Solution** (skip it unless you're stuck)

In [18]:
# Solution
study_system = """You are a study buddy for someone learning to build with the Claude API.
Answer every question with exactly three bullet points and nothing else:
- each bullet is one line that starts with "- "
- each bullet is under 20 words
- no greeting, no introduction, no closing line"""

In [19]:
# Check
def three_short_bullets(text):
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    return len(lines) == 3 and all(ln.startswith("- ") and len(ln[2:].split()) < 20 for ln in lines)


for q in ["Why does the API need max_tokens?", "What is a system prompt for?"]:
    answer = chat([{"role": "user", "content": q}], system=study_system, max_tokens=300)
    print(q)
    print(answer)
    print("✅ three short bullets\n" if three_short_bullets(answer) else "❌ not three short bullets: improve the prompt\n")

Why does the API need max_tokens?
- It caps how many tokens Claude can generate, so a response never runs longer than you intend.
- It controls cost and latency, since output tokens are billed and longer generations take more time.
- If Claude hits the limit, output is cut off mid-thought and stop_reason returns "max_tokens".
✅ three short bullets



What is a system prompt for?
- A system prompt sets Claude's role, tone, and rules before the conversation begins.
- It gives persistent context and instructions that apply across every turn of the exchange.
- Use the API's separate `system` parameter, keeping it distinct from user messages.
✅ three short bullets



## 7. Temperature, and what changed

In the course, `temperature` (0 to 1) controls randomness: low for predictable answers, high for creative ones.

> **Changed since the course.** On current models (Sonnet 5.5, Opus 5.5, Haiku 5.5) a non-default `temperature`, `top_p` or `top_k` is rejected, and from API version `2026-10-21` they are rejected on every model. The current knob is **effort** (`output_config={"effort": "low" | "medium" | "high" | ...}`), which sets how much the model thinks before answering.

The current Python SDK (2.x) has already dropped the argument, so the call fails in Python before anything is sent:

In [20]:
try:
    client.messages.create(
        model=model,
        max_tokens=100,
        temperature=0.0,
        messages=[{"role": "user", "content": "Say hi."}],
    )
except TypeError as e:
    print("Python SDK:", e)

Python SDK: Messages.create() got an unexpected keyword argument 'temperature'


Forcing it into the request body with `extra_body` shows what the API itself says:

In [21]:
import anthropic

try:
    client.messages.create(
        model=model,
        max_tokens=100,
        messages=[{"role": "user", "content": "Say hi."}],
        extra_body={"temperature": 0.0},
    )
except anthropic.BadRequestError as e:
    print("API 400:", e.message)

API 400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'temperature: Extra inputs are not permitted'}, 'request_id': 'req_011Ch67eQXg7eLy3phY3BShL'}


In [22]:
# The current way to trade depth for speed and cost: effort
quick = client.messages.create(
    model=model,
    max_tokens=300,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Give me a one-line name idea for a coffee shop."}],
)
print(text_of(quick), "|", quick.usage.output_tokens, "output tokens")

**"Bean There, Brewed That"** | 16 output tokens


### Exercise 4 · Variety without temperature

In the course, `temperature` set how varied the answers were. It's gone: answers still vary a little from call to call, but you can't turn that up or down. When you need *different* options, ask for them.

**Task.**

1. Send the same request three times: "Suggest one name for a new coffee shop. Reply with just the name." Keep the three names in the list `separate`.
2. Send one request for five *different* names, one per line, and keep them in the list `in_one_call`.

**Done when** the check cell prints ✅: five different names from one call.

In [23]:
# Your turn
separate = []      # three names, from three separate calls
in_one_call = []   # five names, from one call

<details><summary><b>Hint</b></summary>

Use a `for` loop for the three calls, and `text_of(...).strip()` for each name. For the five, ask for "five different names, one per line, nothing else" and split the text with `.splitlines()`. `output_config={"effort": "low"}` keeps these small calls quick.

</details>

**Solution** (skip it unless you're stuck)

In [24]:
# Solution
one_name = "Suggest one name for a new coffee shop. Reply with just the name."
separate = []
for _ in range(3):
    reply = client.messages.create(
        model=model,
        max_tokens=100,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": one_name}],
    )
    separate.append(text_of(reply).strip())

five = client.messages.create(
    model=model,
    max_tokens=200,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Suggest five different names for a new coffee shop, one per line. Output only the names."}],
)
in_one_call = [line.strip() for line in text_of(five).splitlines() if line.strip()]

In [25]:
# Check
print("Three separate calls:", separate, "->", len(set(separate)), "different")
print("One call for five:   ", in_one_call, "->", len(set(in_one_call)), "different")
if len(in_one_call) == 5 and len(set(in_one_call)) == 5:
    print("✅ Asking for variety in the prompt gives you variety.")
else:
    print("❌ Not yet: in_one_call should hold five different names.")

Three separate calls: ['Morning Ritual', 'Daybreak Grounds', 'Brew Haven'] -> 3 different
One call for five:    ['Bean There Brewed', 'The Daily Grind House', 'Ember & Oak Coffee', 'Morning Ritual Café', 'Copper Cup Roasters'] -> 5 different
✅ Asking for variety in the prompt gives you variety.


In this run the three separate calls happened to give three different names, but nothing stops them repeating: each call knows nothing about the others. In one call, Claude sees the names it has already written, so asking for five different ones is what guarantees variety. You can also vary the request itself (a theme, an audience, a style). For *consistent* answers, clear instructions and structured outputs (lessons 2.2 and 2.4) are the tools to reach for.

## Recap

- One endpoint, `messages.create`, with `model`, `max_tokens` and `messages`.
- Read text from the blocks of type `text`; check `stop_reason` and `usage`.
- The API is stateless: resend the whole conversation each time. You own the history (Domain 5).
- The system prompt sets role and rules; rules that must always hold are enforced in code (Domain 4).
- `temperature` is gone on current models; `effort` is the knob now.
- Check `stop_reason`: `max_tokens` means the answer was cut off (exercises 1 and 2).
- Pick the smallest model that does the job well: Haiku, Sonnet, Opus, Fable.

Next: [2.2 Streaming and structured data](02-streaming-and-structured-data.ipynb).